In [1]:
# ===============================================================
# Part 0: 필요한 라이브러리 설치 및 임포트
# ===============================================================
import sys
# !{sys.executable} -m pip install numpy scipy matplotlib neo quantities elephant-toolbox

import numpy as np
import scipy.io
import neo
import quantities as pq
import matplotlib.pyplot as plt
from elephant.gpfa import GPFA
from elephant.conversion import BinnedSpikeTrain

# ===============================================================
# Part 1: 데이터 전처리 (분산 기반 필터링)
# ===============================================================
def preprocess_monkeydata_robust(filepath='monkeydata.mat', bin_size=20*pq.ms):
    """
    Loads monkeydata.mat and applies a robust, variance-based filtering
    on the binned spike data before creating SpikeTrain objects for GPFA.
    """
    print(f"--- Loading data from '{filepath}' ---")
    try:
        data_mat = scipy.io.loadmat(filepath)
    except FileNotFoundError:
        print(f"ERROR: Data file not found at '{filepath}'.")
        return None, 0, 0

    trial_data = data_mat['trial']
    n_trials_per_angle, n_angles = trial_data.shape
    n_neurons_total = trial_data[0, 0]['spikes'].shape[0]
    
    print(f"Found {n_trials_per_angle} trials/angle, {n_angles} angles, {n_neurons_total} initial neurons.")

    # 1. 모든 trial 데이터를 neo.SpikeTrain 객체로 먼저 변환 (필터링 전)
    all_trials_raw = []
    for angle_idx in range(n_angles):
        for trial_idx in range(n_trials_per_angle):
            spike_matrix = trial_data[trial_idx, angle_idx]['spikes']
            n_timesteps = spike_matrix.shape[1]
            trial_spiketrains = []
            for neuron_idx in range(n_neurons_total):
                spike_times = np.where(spike_matrix[neuron_idx, :] == 1)[0] * pq.ms
                spiketrain = neo.SpikeTrain(spike_times, t_stop=n_timesteps * pq.ms)
                trial_spiketrains.append(spiketrain)
            all_trials_raw.append(neo.SpikeTrainList(trial_spiketrains))

    # 2. 데이터를 binning하고, binned data의 분산을 계산
    binned_data = BinnedSpikeTrain(all_trials_raw, bin_size=bin_size)
    # binned_data.to_array() -> (n_bins, n_neurons, n_trials)
    # 분산 계산을 위해 축을 변경: (n_neurons, n_bins * n_trials)
    binned_array = binned_data.to_array().transpose(1, 0, 2).reshape(n_neurons_total, -1)
    variance_per_neuron = np.var(binned_array, axis=1)
    
    # 3. 분산이 0보다 큰, 즉 활동에 변화가 있는 뉴런만 선택
    valid_neuron_indices = np.where(variance_per_neuron > 0)[0]
    n_neurons_active = len(valid_neuron_indices)
    
    if n_neurons_active == 0:
        print("CRITICAL ERROR: No neurons showed any activity after binning. Try a larger bin_size.")
        return None, 0, 0
        
    print(f"Filtering based on variance: {n_neurons_active} of {n_neurons_total} neurons are active at a {bin_size}ms timescale.")
    
    # 4. 유효한 뉴런만 포함하여 최종 데이터 리스트 생성
    all_trials_filtered = []
    for trial in all_trials_raw:
        filtered_spiketrains = [trial[i] for i in valid_neuron_indices]
        all_trials_filtered.append(filtered_spiketrains)
        
    print(f"--- Successfully preprocessed and filtered {len(all_trials_filtered)} trials ---")
    return all_trials_filtered, n_trials_per_angle, n_angles

# ===============================================================
# Part 2: GPFA 실행 및 시각화 (이전과 동일)
# ===============================================================
def run_and_plot_gpfa(all_trials, n_trials_per_angle, n_angles, bin_size=20*pq.ms, x_dim=8):
    if not all_trials:
        return
        
    print(f"\n--- Running GPFA on {len(all_trials)} total trials ---")
    gpfa = GPFA(bin_size=bin_size, x_dim=x_dim)
    trajectories = gpfa.fit_transform(all_trials)
    print("--- GPFA Complete ---")

    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection='3d')
    colors = plt.cm.jet(np.linspace(0, 1, n_angles))

    for trial_idx, traj in enumerate(trajectories):
        angle_idx = trial_idx // n_trials_per_angle
        ax.plot(traj[0, :], traj[1, :], traj[2, :], color=colors[angle_idx], alpha=0.4)

    ax.set_xlabel('Latent Dimension 1')
    ax.set_ylabel('Latent Dimension 2')
    ax.set_zlabel('Latent Dimension 3')
    ax.set_title('GPFA Latent Trajectories by Reaching Angle')
    plt.show()

# ===============================================================
# 메인 실행 블록
# ===============================================================


In [2]:
BIN_SIZE = 20 * pq.ms # 분석의 시간 해상도 (조절 가능)

all_trials, n_trials, n_angles = preprocess_monkeydata_robust(bin_size=BIN_SIZE)

run_and_plot_gpfa(all_trials, n_trials, n_angles, bin_size=BIN_SIZE)

--- Loading data from 'monkeydata.mat' ---
Found 100 trials/angle, 8 angles, 98 initial neurons.


AttributeError: module 'neo' has no attribute 'SpikeTrainList'